### Importing basic and modelling libraries

In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline 
import seaborn as sns 

from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer 
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso 
from sklearn.neighbors import KNeighborsRegressor

import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

### Importing Data as Pandas DataFrame

In [2]:
df = pd.read_csv('data/stud.csv')

### Show Top 5 Records

In [3]:
df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


### Preparing X and Y variables

In [4]:
X = df.drop('math_score', axis=1)
y = df['math_score']

### Feature Engineering
Creating Column Transformer for converting categorical features' values in numbers and scaling numerical features as pipeline

In [5]:
numeric_features = X.select_dtypes(exclude='str').columns
categorical_features = X.select_dtypes(include='str').columns

numeric_transformer = RobustScaler()
ohe_transformer = OneHotEncoder()

preprocessor = ColumnTransformer(
  [
    ("OneHotEncoder", ohe_transformer, categorical_features),
    ("RobustScaler", numeric_transformer, numeric_features)
  ]
)
X = preprocessor.fit_transform(X)
X

array([[ 1.        ,  0.        ,  0.        , ...,  1.        ,
         0.1       ,  0.23529412],
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,
         1.        ,  0.89411765],
       [ 1.        ,  0.        ,  0.        , ...,  1.        ,
         1.25      ,  1.12941176],
       ...,
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,
         0.05      , -0.18823529],
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,
         0.4       ,  0.37647059],
       [ 1.        ,  0.        ,  0.        , ...,  1.        ,
         0.8       ,  0.8       ]], shape=(1000, 19))

### Separate dataset into Train and Test

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

((800, 19), (200, 19))

### Creating Evaluate function to return r2_score and mae metrics of all models

In [7]:
def evaluate_models(true, predicted):
  mae = mean_absolute_error(true, predicted)
  r2 = r2_score(true, predicted)
  
  return r2, mae

### Training and Evaluating the different models to find best model

In [9]:
models = {
  'LinearRegression': LinearRegression(),
  'Ridge': Ridge(),
  'Lasso': Lasso(),
  'DecisionTreeRegressor': DecisionTreeRegressor(),
  'RandomForestRegressor': RandomForestRegressor(),
  'KNeighborsRegressor': KNeighborsRegressor()
}

model_list = []
r2_score_list = []
mae_list = []

for i in range(len(list(models))):
  model = list(models.values())[i]
  model.fit(X_train, y_train)  # Training the model
  
  # Making predictions 
  y_train_pred = model.predict(X_train)
  y_test_pred = model.predict(X_test) 
  
  # Evaluate Train and Test dataset 
  model_train_r2_score, model_train_mae = evaluate_models(y_train, y_train_pred)
  model_test_r2_score, model_test_mae = evaluate_models(y_test, y_test_pred)
  
  model_list.append(list(models.keys())[i])
  r2_score_list.append(model_test_r2_score)
  mae_list.append(model_test_mae)
  
  print(list(models.keys())[i])
  print('Model performance for Training set')
  print('-R2 Score: {:.4f}'.format(model_train_r2_score))
  print('-Mean Abosulute Error: {:.4f}'.format(model_train_mae))
  
  print('Model performance for Test set')
  print('-R2 Score: {:.4f}'.format(model_test_r2_score))
  print('-Mean Abosulute Error: {:.4f}'.format(model_test_mae))
  
  print('='*35)
  print('\n')

LinearRegression
Model performance for Training set
-R2 Score: 0.8743
-Mean Abosulute Error: 4.2667
Model performance for Test set
-R2 Score: 0.8804
-Mean Abosulute Error: 4.2148


Ridge
Model performance for Training set
-R2 Score: 0.8743
-Mean Abosulute Error: 4.2638
Model performance for Test set
-R2 Score: 0.8807
-Mean Abosulute Error: 4.2100


Lasso
Model performance for Training set
-R2 Score: 0.8016
-Mean Abosulute Error: 5.2910
Model performance for Test set
-R2 Score: 0.8192
-Mean Abosulute Error: 5.2517


DecisionTreeRegressor
Model performance for Training set
-R2 Score: 0.9997
-Mean Abosulute Error: 0.0187
Model performance for Test set
-R2 Score: 0.7245
-Mean Abosulute Error: 6.4250


RandomForestRegressor
Model performance for Training set
-R2 Score: 0.9765
-Mean Abosulute Error: 1.8497
Model performance for Test set
-R2 Score: 0.8489
-Mean Abosulute Error: 4.6999


KNeighborsRegressor
Model performance for Training set
-R2 Score: 0.8212
-Mean Abosulute Error: 5.0067
Mode

### Choosing best model based on r2_score and mae

In [10]:
pd.DataFrame(list(zip(model_list, r2_score_list, mae_list)), columns= ['Model Name', 'R2 Score', 'MAE']).sort_values(by='R2 Score', ascending=False)

,Model Name,R2 Score,MAE
1,Ridge,0.880698,4.209995
0,LinearRegression,0.880433,4.214763
4,RandomForestRegressor,0.848860,4.699879
2,Lasso,0.819235,5.251743
5,KNeighborsRegressor,0.730526,6.269000
3,DecisionTreeRegressor,0.724478,6.425000


`Insights`: We can see that `Ridge` performed well with an accuracy of `88%` and on average the model is `off by
            `4.2` points.

## Ridge: Best model

In [11]:
ridge_model = Ridge()
ridge_model.fit(X_train, y_train)
y_pred = ridge_model.predict(X_test)
score = r2_score(y_test, y_pred)*100

print("Accuracy of the model is %.2f" %score)

Accuracy of the model is 88.07


### Difference between Actual and Predicted Values

In [12]:
pred_df = pd.DataFrame(
  {
    'Actual Value': y_test,
    'Predicted Value': y_pred,
    'Difference': y_test-y_pred
  }
)
pred_df

,Actual Value,Predicted Value,Difference
521,91,76.398367,14.601633
737,53,58.789434,-5.789434
740,80,76.999530,3.000470
660,74,76.787582,-2.787582
411,84,87.647180,-3.647180
...,...,...,...
408,52,43.528434,8.471566
332,62,62.215611,-0.215611
208,74,67.918926,6.081074
613,65,67.080774,-2.080774
